In [1]:
import os
if os.getcwd().endswith("notebooks"):
    os.chdir(os.getcwd() + "/..")
os.getcwd()

'/Users/francois-xavierfabre/workdir/github/parse_sql'

In [2]:
import re
import pandas as pd
from pathlib import Path
import sqlfluff
import json
from pprint import pprint
from collections import Counter
from contextlib import suppress
import os

from parse_sql.parsing_tools import parse_raw_query, get
from parse_sql.from_join import extract_from_join
from parse_sql.parsers import read_and_parse_sql_file, extract_select
from parse_sql.extract_from_parsed import get_bq_schema_with_cols, extract_all_joins_from_file

pd.set_option("display.width", 250)

def pretty_print(json_content):
    print(json.dumps(json_content, indent=4))

# Extract info from SQL

In [3]:
dag_dir = Path(os.getenv("WORK_DIR")) / "data-flow/dags" / "d03_integration"
dag_dir

PosixPath('/Users/francois-xavierfabre/workdir/effy/data-flow/dags/d03_integration')

In [4]:
def iter_files(start_folder: Path):
    for root, dirs, files in os.walk(start_folder.as_posix()):
        root_folder = Path(root).absolute()
        for file_name in files:
            if not file_name.endswith(".sql"):
                continue
            yield root_folder / file_name


c = Counter()
for file_path in iter_files(dag_dir):
    print(file_path.as_posix())
    try:
        parsing_by_cte = read_and_parse_sql_file(file_path)
        all_joins = extract_all_joins_from_file(parsing_by_cte)
        c.update(all_joins)
    except Exception as e:
        print("  Failed :", e)

c.most_common(30)

/Users/francois-xavierfabre/workdir/effy/data-flow/dags/d03_integration/sql/sherlock/DW_SHERLOCK/ods_to_dw_campagnes.sql
/Users/francois-xavierfabre/workdir/effy/data-flow/dags/d03_integration/sql/sherlock/DW_SHERLOCK/ods_to_dw_operateurs.sql
  Guessing last_state.id is from ODS_SHERLOCK.operateurs_raw
/Users/francois-xavierfabre/workdir/effy/data-flow/dags/d03_integration/sql/sherlock/DW_SHERLOCK/ods_to_dw_solutions.sql
  Guessing last_state.id is from ODS_SHERLOCK.solutions_raw
/Users/francois-xavierfabre/workdir/effy/data-flow/dags/d03_integration/sql/sherlock/DW_SHERLOCK/ods_to_dw_pistes_histo.sql
  Guessing last_state_campagne.id is from ODS_SHERLOCK.diabolocom_campagnes_raw
ERROR : Unable to find id in cte campagnes
/Users/francois-xavierfabre/workdir/effy/data-flow/dags/d03_integration/sql/sherlock/DW_SHERLOCK/ods_to_dw_opportunites.sql
  ERROR : Unable to resolve col first_winning_date_lf.id
ERROR : Unable to find id in cte last_five_days
  ERROR : Unable to resolve col first_d

[('function() = ODS_DIABOLO.contacts_raw:_PARTITIONDATE', 4),
 ('ODS_DIABOLO.contacts_raw:_PARTITIONDATE = function()', 4),
 ('function() = ODS_DIABOLO.calls_details_recording_raw:extract_datetime', 4),
 ('ODS_DIABOLO.calls_details_recording_raw:extract_datetime = function()', 4),
 ('function() = ODS_DIABOLO.calls_details_recording_raw:_PARTITIONDATE', 4),
 ('ODS_DIABOLO.calls_details_recording_raw:_PARTITIONDATE = function()', 4),
 ('ODS_SHERLOCK.diabolocom_campagnes_raw:updated_at = function()', 2),
 ('function() = ODS_SHERLOCK.diabolocom_campagnes_raw:updated_at', 2),
 ('ODS_SHERLOCK.parametre_pistes_raw:updated_at = function()', 2),
 ('function() = ODS_SHERLOCK.parametre_pistes_raw:updated_at', 2),
 ('DW_PRIMEO.document_collections:project_id = DW_PRIMEO.events:project_id',
  2),
 ('DW_PRIMEO.events:project_id = DW_PRIMEO.document_collections:project_id',
  2),
 ('DW_PRIMEO.events:id = function()', 2),
 ('function() = DW_PRIMEO.events:id', 2),
 ('ODS_PRIMEO.parameters_raw:updated_a

In [ ]:
[
    f"{join_cond.split(' = ')[0]:>50} {join_cond.split(' = ')[1]:<50} : {count}"
    for join_cond, count in c.most_common(50)
    if (join_cond.split(" = ")[0] != join_cond.split(" = ")[1]) and (join_cond.split(' = ')[0] != "function()") and (join_cond.split(' = ')[1] != "function()")
]

# Sandbox

In [ ]:
query = Path("query_cte.sql").read_text()
parsing_by_cte = read_and_parse_sql_file(query)

for cte_name, cte_query in parsing_by_cte.items():
    print(f"=== {cte_name} ===")
    pprint(cte_query)
    print()

In [ ]:
parse_raw_query(query)